# SQL to PySpark Phase 2: Revised Bridge Pack

**Objective:** Bridge the gap between basic SQL-to-PySpark syntax and real-world data engineering tasks. Work with sample datasets, perform light data cleaning, and solve realistic joins and aggregations by comparing SQL logic directly with PySpark Transformations.

## 1. Environment Setup & Data Generation

In [1]:
!pip install -q pyspark

import os

customers_data = '''customer_id,first_name,last_name,email,city
1,John,Smith,john@domain.com,Springfield
2,Emma,Jones,emma@webmail.com,Centerville
3,Olivia,Brown,olivia@outlook.com,Greenville
4,Liam,Johnson,liam@gmail.com,Riverside
5,Noah,Williams,noah@yahoo.com,Lakeside
6,Alice,Miller,alice@aol.com,Oakland
,Invalid,User,invalid@aol.com,Nowhere
'''

orders_data = '''order_id,customer_id,order_date,amount
1,1,2024-01-15,39.98
2,1,2024-01-20,29.99
3,2,2024-01-16,25.00
4,2,2024-01-22,89.97
5,3,2024-01-17,49.98
6,4,2024-01-18,119.96
7,4,2024-01-25,15.50
8,5,2024-01-19,66.75
9,1,2024-02-15,22.19
10,4,2024-02-20,59.98
11,2,2024-02-22,22.25
12,,2024-03-01,99.99
'''

with open('customers.csv', 'w') as f: f.write(customers_data)
with open('orders.csv', 'w') as f: f.write(orders_data)

print('Data files generated successfully!')


Data files generated successfully!


## 2. Load Data & Mini Cleaning Task

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, desc

spark = SparkSession.builder.appName("Phase2-BridgePack").getOrCreate()

# Load Data
customers = spark.read.option("header", "true").option("inferSchema", "true").csv("customers.csv")
orders = spark.read.option("header", "true").option("inferSchema", "true").csv("orders.csv")

print("--- Raw Data ---")
customers.show(3)
orders.show(3)

# Mini Cleaning Task: Remove rows with missing customer_id
customers = customers.dropna(subset=["customer_id"])
orders = orders.dropna(subset=["customer_id"])

print("--- Cleaned Data ---")
print(f"Customers count: {customers.count()}")
print(f"Orders count: {orders.count()}")

# Register as Temp Views for SQL exercises
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")


--- Raw Data ---
+-----------+----------+---------+------------------+-----------+
|customer_id|first_name|last_name|             email|       city|
+-----------+----------+---------+------------------+-----------+
|          1|      John|    Smith|   john@domain.com|Springfield|
|          2|      Emma|    Jones|  emma@webmail.com|Centerville|
|          3|    Olivia|    Brown|olivia@outlook.com| Greenville|
+-----------+----------+---------+------------------+-----------+
only showing top 3 rows
+--------+-----------+----------+------+
|order_id|customer_id|order_date|amount|
+--------+-----------+----------+------+
|       1|          1|2024-01-15| 39.98|
|       2|          1|2024-01-20| 29.99|
|       3|          2|2024-01-16|  25.0|
+--------+-----------+----------+------+
only showing top 3 rows
--- Cleaned Data ---
Customers count: 6
Orders count: 11


## 3. Guided Exercises (SQL vs PySpark)

### Exercise 1: Total order amount for each customer

In [3]:
print("--- SQL Approach ---")
sql_1 = spark.sql("""
    SELECT customer_id, SUM(amount) AS total_amount
    FROM orders
    GROUP BY customer_id
    ORDER BY customer_id
""")
sql_1.show()

print("--- PySpark Approach ---")
pyspark_1 = orders.groupBy("customer_id") \
    .agg(sum("amount").alias("total_amount")) \
    .orderBy("customer_id")
pyspark_1.show()


--- SQL Approach ---
+-----------+------------------+
|customer_id|      total_amount|
+-----------+------------------+
|          1|             92.16|
|          2|            137.22|
|          3|             49.98|
|          4|195.43999999999997|
|          5|             66.75|
+-----------+------------------+

--- PySpark Approach ---
+-----------+------------------+
|customer_id|      total_amount|
+-----------+------------------+
|          1|             92.16|
|          2|            137.22|
|          3|             49.98|
|          4|195.43999999999997|
|          5|             66.75|
+-----------+------------------+



### Exercise 2: Top 3 customers by total spend

In [4]:
print("--- SQL Approach ---")
sql_2 = spark.sql("""
    SELECT customer_id, SUM(amount) AS total_spend
    FROM orders
    GROUP BY customer_id
    ORDER BY total_spend DESC
    LIMIT 3
""")
sql_2.show()

print("--- PySpark Approach ---")
pyspark_2 = orders.groupBy("customer_id") \
    .agg(sum("amount").alias("total_spend")) \
    .orderBy(desc("total_spend")) \
    .limit(3)
pyspark_2.show()


--- SQL Approach ---
+-----------+------------------+
|customer_id|       total_spend|
+-----------+------------------+
|          4|195.43999999999997|
|          2|            137.22|
|          1|             92.16|
+-----------+------------------+

--- PySpark Approach ---
+-----------+------------------+
|customer_id|       total_spend|
+-----------+------------------+
|          4|195.43999999999997|
|          2|            137.22|
|          1|             92.16|
+-----------+------------------+



### Exercise 3: Customers with no orders

In [5]:
print("--- SQL Approach ---")
sql_3 = spark.sql("""
    SELECT c.customer_id, c.first_name
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.order_id IS NULL
""")
sql_3.show()

print("--- PySpark Approach ---")
pyspark_3 = customers.join(orders, "customer_id", "left") \
    .filter(col("order_id").isNull()) \
    .select(customers["customer_id"], "first_name")
pyspark_3.show()


--- SQL Approach ---
+-----------+----------+
|customer_id|first_name|
+-----------+----------+
|          6|     Alice|
+-----------+----------+

--- PySpark Approach ---
+-----------+----------+
|customer_id|first_name|
+-----------+----------+
|          6|     Alice|
+-----------+----------+



### Exercise 4: City-wise total revenue

In [6]:
print("--- SQL Approach ---")
sql_4 = spark.sql("""
    SELECT c.city, ROUND(SUM(o.amount), 2) AS total_revenue
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.city
    ORDER BY total_revenue DESC
""")
sql_4.show()

print("--- PySpark Approach ---")
from pyspark.sql.functions import round

pyspark_4 = customers.join(orders, "customer_id", "inner") \
    .groupBy("city") \
    .agg(round(sum("amount"), 2).alias("total_revenue")) \
    .orderBy(desc("total_revenue"))
pyspark_4.show()


--- SQL Approach ---
+-----------+-------------+
|       city|total_revenue|
+-----------+-------------+
|  Riverside|       195.44|
|Centerville|       137.22|
|Springfield|        92.16|
|   Lakeside|        66.75|
| Greenville|        49.98|
+-----------+-------------+

--- PySpark Approach ---
+-----------+-------------+
|       city|total_revenue|
+-----------+-------------+
|  Riverside|       195.44|
|Centerville|       137.22|
|Springfield|        92.16|
|   Lakeside|        66.75|
| Greenville|        49.98|
+-----------+-------------+



### Exercise 5: Average order amount per customer

In [7]:
print("--- SQL Approach ---")
sql_5 = spark.sql("""
    SELECT customer_id, ROUND(AVG(amount), 2) AS avg_amount
    FROM orders
    GROUP BY customer_id
    ORDER BY customer_id
""")
sql_5.show()

print("--- PySpark Approach ---")
pyspark_5 = orders.groupBy("customer_id") \
    .agg(round(avg("amount"), 2).alias("avg_amount")) \
    .orderBy("customer_id")
pyspark_5.show()


--- SQL Approach ---
+-----------+----------+
|customer_id|avg_amount|
+-----------+----------+
|          1|     30.72|
|          2|     45.74|
|          3|     49.98|
|          4|     65.15|
|          5|     66.75|
+-----------+----------+

--- PySpark Approach ---
+-----------+----------+
|customer_id|avg_amount|
+-----------+----------+
|          1|     30.72|
|          2|     45.74|
|          3|     49.98|
|          4|     65.15|
|          5|     66.75|
+-----------+----------+



### Exercise 6: Customers with more than one order

In [8]:
print("--- SQL Approach ---")
sql_6 = spark.sql("""
    SELECT customer_id, COUNT(order_id) AS order_count
    FROM orders
    GROUP BY customer_id
    HAVING COUNT(order_id) > 1
    ORDER BY order_count DESC
""")
sql_6.show()

print("--- PySpark Approach ---")
pyspark_6 = orders.groupBy("customer_id") \
    .agg(count("order_id").alias("order_count")) \
    .filter(col("order_count") > 1) \
    .orderBy(desc("order_count"))
pyspark_6.show()


--- SQL Approach ---
+-----------+-----------+
|customer_id|order_count|
+-----------+-----------+
|          1|          3|
|          4|          3|
|          2|          3|
+-----------+-----------+

--- PySpark Approach ---
+-----------+-----------+
|customer_id|order_count|
+-----------+-----------+
|          1|          3|
|          4|          3|
|          2|          3|
+-----------+-----------+



### Exercise 7: Sort customers by total spend descending

In [9]:
print("--- SQL Approach ---")
sql_7 = spark.sql("""
    SELECT c.customer_id, c.first_name, c.last_name, ROUND(SUM(o.amount), 2) AS total_spend
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.first_name, c.last_name
    ORDER BY total_spend DESC
""")
sql_7.show()

print("--- PySpark Approach ---")
pyspark_7 = customers.join(orders, "customer_id", "inner") \
    .groupBy(customers["customer_id"], "first_name", "last_name") \
    .agg(round(sum("amount"), 2).alias("total_spend")) \
    .orderBy(desc("total_spend"))
pyspark_7.show()


--- SQL Approach ---
+-----------+----------+---------+-----------+
|customer_id|first_name|last_name|total_spend|
+-----------+----------+---------+-----------+
|          4|      Liam|  Johnson|     195.44|
|          2|      Emma|    Jones|     137.22|
|          1|      John|    Smith|      92.16|
|          5|      Noah| Williams|      66.75|
|          3|    Olivia|    Brown|      49.98|
+-----------+----------+---------+-----------+

--- PySpark Approach ---
+-----------+----------+---------+-----------+
|customer_id|first_name|last_name|total_spend|
+-----------+----------+---------+-----------+
|          4|      Liam|  Johnson|     195.44|
|          2|      Emma|    Jones|     137.22|
|          1|      John|    Smith|      92.16|
|          5|      Noah| Williams|      66.75|
|          3|    Olivia|    Brown|      49.98|
+-----------+----------+---------+-----------+

